# Section 07 Onboarding Notebook: Quantum Dynamics — 2D MCTDH, 1D TD-DMRG, and Open-System HEOM

### Overview & Pedagogical Design
When nuclear tunneling, zero-point energy, conical intersections, or strong non-Markovian bath memory govern chemical dynamics, classical trajectories are no longer sufficient and we must solve the time-dependent quantum equations of motion directly. This notebook covers the three flagship quantum dynamics methods used in modern chemical physics:
1. **Multi-Configuration Time-Dependent Hartree (MCTDH)** ([J. Chem. Phys. 154, 114107 (2021)](https://pubs.aip.org/aip/jcp/article/154/11/114107/315442/A-rectangular-collocation-multi-configuration-time)): Compresses a multi-dimensional nuclear wavepacket $\Psi(q_1, q_2, t)$ on a Discrete Variable Representation (DVR) grid into a low-rank Tucker expansion of **time-dependent Single-Particle Functions (SPFs)** $\varphi_j^{(\kappa)}(q_\kappa, t)$ evolved via the **Dirac–Frenkel variational principle**.
2. **Time-Dependent Density Matrix Renormalization Group (TD-DMRG / TEBD)** ([J. Chem. Theory Comput. 14, 5024 (2018)](https://pubs.acs.org/doi/abs/10.1021/acs.jctc.8b00628)): Evolves 1D exciton–polaron chains as a **Matrix Product State (MPS)** using second-order **Suzuki–Trotter (Strang) unitary gates** and SVD bond compression.
3. **Hierarchical Equations of Motion (HEOM)** ([QuTiP HEOM Guide](https://qutip.org/docs/4.7/guide/guide-heom.html)): Propagates the reduced density matrix $\hat{\rho}_0(t)$ coupled to a non-Markovian Drude–Lorentz thermal bath **non-perturbatively and non-Markovianly** via a hierarchy of **Auxiliary Density Operators (ADOs)** $\hat{\rho}_n(t)$.

In this notebook, **no MCTDH, TD-DMRG, or HEOM functions are pre-solved for you**. You will implement **18 granular `# TODO` functions** (6 per method):
- **Part 1 — 2D MCTDH Wavepacket Dynamics (`TODO 1` – `TODO 6`)**
- **Part 2 — 1D TD-DMRG / MPS Exciton–Polaron Transport (`TODO 7` – `TODO 12`)**
- **Part 3 — Tanimura–Kubo HEOM Non-Markovian Spin–Boson Dynamics (`TODO 13` – `TODO 18`)**
- **Part 4 — Step-by-Step Checkpoint & Verification Suite (`run_section07_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, expm, qr, svd
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
print("Section 07 Quantum Dynamics environment initialized. Implement TODOs 1-18 and run `run_section07_checkpoints()`!")

## Part 1: 2D Multi-Configuration Time-Dependent Hartree (MCTDH) Wavepacket Dynamics (TODOs 1–6)

### Mathematical Specifications
For a 2D bilinearly coupled anharmonic oscillator $\hat{H}(q_1, q_2) = \hat{h}_1(q_1) + \hat{h}_2(q_2) + c_{\text{coup}}\,\hat{q}_1 \hat{q}_2$, the MCTDH ansatz expands the wavepacket in $n_{\text{spf}}$ time-dependent orthonormal Single-Particle Functions (SPFs) per mode:
$$\Psi(q_1, q_2, t) = \sum_{j_1=1}^{n_{\text{spf}}}\sum_{j_2=1}^{n_{\text{spf}}} A_{j_1 j_2}(t)\,\varphi_{j_1}^{(1)}(q_1, t)\,\varphi_{j_2}^{(2)}(q_2, t)$$

1. **1D Sine-DVR Grid & Kinetic Energy Matrix (`build_1d_sine_dvr_grid_and_kinetic` — TODO 1)**:
   On $N$ interior points $q_\alpha = q_{\min} + \alpha\,\Delta q$ ($\alpha = 1, \dots, N$, with $\Delta q = (q_{\max} - q_{\min})/(N+1)$ and box length $L = q_{\max} - q_{\min}$):
   $$T_{\alpha\beta} = \frac{\pi^2}{4 m L^2} \begin{cases} \frac{2(N+1)^2 + 1}{3} - \frac{1}{\sin^2\!\left(\frac{\pi\alpha}{N+1}\right)} & \alpha = \beta \\ (-1)^{\alpha-\beta}\left[\frac{1}{\sin^2\!\left(\frac{\pi(\alpha-\beta)}{2(N+1)}\right)} - \frac{1}{\sin^2\!\left(\frac{\pi(\alpha+\beta)}{2(N+1)}\right)}\right] & \alpha \ne \beta \end{cases}$$

2. **Displaced Gaussian Initialization & SPF Orthonormalization (`initialize_2d_mctdh_state` — TODO 2)**:
   Build initial orthonormal SPF matrices $\boldsymbol{\varphi}^{(1)}, \boldsymbol{\varphi}^{(2)} \in \mathbb{C}^{N_{\text{grid}} \times n_{\text{spf}}}$ via QR decomposition where column $0$ is the normalized displaced Gaussian $\chi_0(q_\kappa) \propto e^{-\omega_\kappa(q_\kappa - q_{\kappa,0})^2/2}$, and set $A_{0,0}(0) = 1.0$ (all other $A_{j_1 j_2}(0) = 0$).

3. **Single-Particle Density Matrices & Regularized Inverses (`compute_mctdh_density_and_regularized_inv` — TODO 3)**:
   $$\rho^{(1)} = \mathbf{A}\mathbf{A}^\dagger, \qquad \rho^{(2)} = \mathbf{A}^\mathsf{T}\mathbf{A}^*, \qquad \left(\rho^{(\kappa)}_{\text{reg}}\right)^{-1} = \mathbf{U}\left(\boldsymbol{\lambda} + \epsilon_{\text{reg}} e^{-\boldsymbol{\lambda}/\epsilon_{\text{reg}}}\right)^{-1}\mathbf{U}^\dagger$$

4. **SPF-Space Operators, Mean Fields & Tangent-Space Projectors (`compute_mctdh_projectors_and_mean_fields` — TODO 4)** & **Dirac–Frenkel RHS (`evaluate_mctdh_dirac_frenkel_rhs` — TODO 5)**:
   $$\begin{aligned}
   i\dot{\mathbf{A}} &= \mathbf{h}_{\text{spf}}^{(1)}\mathbf{A} + \mathbf{A}\left(\mathbf{h}_{\text{spf}}^{(2)}\right)^\mathsf{T} + c_{\text{coup}}\,\mathbf{q}_{\text{spf}}^{(1)}\,\mathbf{A}\left(\mathbf{q}_{\text{spf}}^{(2)}\right)^\mathsf{T} \\
   i\dot{\boldsymbol{\varphi}}^{(1)} &= \left(\mathbf{I} - \boldsymbol{\varphi}^{(1)}\boldsymbol{\varphi}^{(1)\dagger}\right)\left[ \mathbf{h}_1\boldsymbol{\varphi}^{(1)} + c_{\text{coup}}\,\mathbf{q}_1\boldsymbol{\varphi}^{(1)}\,\mathbf{M}^{(1)}\left(\rho_{\text{reg}}^{(1)}\right)^{-1} \right]
   \end{aligned}$$
   where $\mathbf{M}^{(1)} = \mathbf{A}^*\,\mathbf{q}_{\text{spf}}^{(2)}\,\mathbf{A}^\mathsf{T}$ (and symmetrically for mode 2 with $\mathbf{M}^{(2)} = \mathbf{A}^\dagger\,\mathbf{q}_{\text{spf}}^{(1)}\,\mathbf{A}$).

5. **2D MCTDH RK4 Propagation & Autocorrelation $C(t)$ (`run_2d_mctdh_simulation` — TODO 6)**:
   Evaluate the exact wavepacket overlap $C(t) = \langle \Psi(0) | \Psi(t) \rangle = \text{Tr}\!\left[ \mathbf{A}(0)^\dagger \left(\boldsymbol{\varphi}^{(1)}(0)^\dagger\boldsymbol{\varphi}^{(1)}(t)\right) \mathbf{A}(t) \left(\boldsymbol{\varphi}^{(2)}(0)^\dagger\boldsymbol{\varphi}^{(2)}(t)\right)^\mathsf{T} \right]$.

In [ ]:
def build_1d_sine_dvr_grid_and_kinetic(
    n_grid: int, q_min: float = -4.0, q_max: float = 4.0, mass: float = 1.0
) -> tuple[np.ndarray, np.ndarray, float]:
    '''
    TODO 1: Construct the 1D Sine-DVR grid points `q` (shape n_grid), kinetic energy matrix `T`
    (shape n_grid x n_grid), and grid spacing `dq`. Return (q, T, dq).
    '''
    # TODO 1: Build interior points i = 1..n_grid and evaluate the exact Colbert-Miller Sine-DVR matrix T.
    raise NotImplementedError("Complete TODO 1: build_1d_sine_dvr_grid_and_kinetic")


def initialize_2d_mctdh_state(
    q1: np.ndarray,
    q2: np.ndarray,
    n_spf: int = 4,
    omega1: float = 1.0,
    omega2: float = 1.2,
    q1_shift: float = 0.8,
    q2_shift: float = -0.6,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 2: Initialize core tensor `A` of shape (n_spf, n_spf) and orthonormal SPF matrices
    `phi1`, `phi2` of shape (n_grid, n_spf) (complex128), where column 0 of phi_k is the
    normalized displaced Gaussian wavepacket and columns 1..n_spf-1 are higher polynomial states
    orthonormalized via QR decomposition (fixing QR sign so column 0 matches the Gaussian).
    Return (A, phi1, phi2).
    '''
    # TODO 2: Construct displaced Gaussians, form polynomial basis (q - q_shift)^k * g0, orthonormalize via qr,
    #         and set A[0, 0] = 1.0 + 0.0j.
    raise NotImplementedError("Complete TODO 2: initialize_2d_mctdh_state")


def compute_mctdh_density_and_regularized_inv(
    A: np.ndarray, reg_eps: float = 1e-8
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 3: From core tensor A (n_spf, n_spf), compute single-particle density matrices
    rho1 = A @ A^dagger, rho2 = A^T @ A^* and their Tikhonov-regularized inverses
    inv_rho1, inv_rho2. Return (rho1, rho2, inv_rho1, inv_rho2).
    '''
    # TODO 3: Diagonalize rho1 and rho2 with eigh, apply w_reg = w + reg_eps * exp(-w / reg_eps),
    #         and reconstruct inv_rho = U @ diag(1 / w_reg) @ U^dagger.
    raise NotImplementedError("Complete TODO 3: compute_mctdh_density_and_regularized_inv")


def compute_mctdh_projectors_and_mean_fields(
    A: np.ndarray,
    phi1: np.ndarray,
    phi2: np.ndarray,
    h1: np.ndarray,
    h2: np.ndarray,
    q1_op: np.ndarray,
    q2_op: np.ndarray,
) -> dict[str, np.ndarray]:
    '''
    TODO 4: Compute SPF-projected 1D operators (h1_spf, h2_spf, q1_spf, q2_spf),
    coupling mean-field matrices M1 = A^* @ q2_spf @ A^T and M2 = A^dagger @ q1_spf @ A,
    and orthogonal tangent-space projectors P1_orth = I - phi1 @ phi1^dagger and P2_orth = I - phi2 @ phi2^dagger.
    Return a dict with keys 'h1_spf', 'h2_spf', 'q1_spf', 'q2_spf', 'M1', 'M2', 'P1_orth', 'P2_orth'.
    '''
    # TODO 4: Compute all 8 matrices and return in a dictionary.
    raise NotImplementedError("Complete TODO 4: compute_mctdh_projectors_and_mean_fields")


def evaluate_mctdh_dirac_frenkel_rhs(
    A: np.ndarray,
    phi1: np.ndarray,
    phi2: np.ndarray,
    h1: np.ndarray,
    h2: np.ndarray,
    q1_op: np.ndarray,
    q2_op: np.ndarray,
    c_coup: float = 0.20,
    reg_eps: float = 1e-8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    '''
    TODO 5: Evaluate the Dirac-Frenkel time derivatives (dA_dt, dphi1_dt, dphi2_dt) and
    total energy expectation value <H> using TODOs 3 and 4.
    '''
    # TODO 5: Combine TODO 3 and TODO 4 to compute H_A, dA_dt = -1j * H_A, <H> = Re(vdot(A, H_A)),
    #         dphi1_dt, and dphi2_dt.
    raise NotImplementedError("Complete TODO 5: evaluate_mctdh_dirac_frenkel_rhs")


def run_2d_mctdh_simulation(
    n_spf: int = 4,
    n_grid: int = 24,
    dt: float = 0.02,
    n_steps: int = 120,
    c_coup: float = 0.20,
) -> dict[str, np.ndarray]:
    '''
    TODO 6: Propagate the 2D MCTDH wavepacket for `n_steps` steps using RK4 (re-orthonormalizing
    phi1, phi2 via QR at each step while absorbing R_1, R_2 into A -> R_1 @ A @ R_2^T).
    Return a dict with keys 'times', 'autocorrelation', 'norm_history', 'energy_history', 'nat_pops_mode1'.
    '''
    # TODO 6: Initialize via TODO 1 & 2, integrate with RK4 using TODO 5, and record C(t), norm, energy, and eigvalsh(A A^dagger).
    raise NotImplementedError("Complete TODO 6: run_2d_mctdh_simulation")

## Part 2: 1D Time-Dependent DMRG (TD-DMRG / TEBD) for Exciton–Polaron Transport (TODOs 7–12)

### Mathematical Specifications
We model coherent exciton/charge transport along an $L$-site 1D molecular aggregate with nearest-neighbor hopping $J_{\text{hop}}$ and nearest-neighbor vibronic/Coulomb interaction $V_{\text{int}}$ ([TD-DMRG Polaron JCTC](https://pubs.acs.org/doi/abs/10.1021/acs.jctc.8b00628)):
$$\hat{H}_{j, j+1} = -J_{\text{hop}}\left(\hat{\sigma}_j^+ \hat{\sigma}_{j+1}^- + \hat{\sigma}_j^- \hat{\sigma}_{j+1}^+\right) + V_{\text{int}}\,\hat{n}_j \hat{n}_{j+1}$$
1. **Localized Product MPS (`initialize_localized_exciton_mps` — TODO 7)**: Initialize rank-3 tensors $A^{[\ell]}$ of shape `(1, 2, 1)` with $|1\rangle$ at `excited_site` and $|0\rangle$ elsewhere.
2. **Two-Site Unitary Suzuki–Trotter Gates (`build_two_site_trotter_gates` — TODO 8)**: Exponentiate $\hat{h}_{2\text{site}}$ via `scipy.linalg.expm(-1j * h_2site * tau)` for $\tau = \Delta t/2$ and $\tau = \Delta t$, reshaping into `(2, 2, 2, 2)`.
3. **Two-Site Gate Application & SVD Compression (`apply_two_site_tebd_gate_and_svd` — TODO 9)**: Contract $A^{[\ell]}, A^{[\ell+1]}$ with $U$, apply SVD truncation, compute bipartite entanglement entropy $S_{\text{vN}}$, and split $\sqrt{\mathbf{S}}$ symmetrically into `(new_left, new_right)`.
4. **Second-Order Strang–Trotter Sweep (`perform_strang_trotter_sweep` — TODO 10)**: Apply $\Delta t/2$ on odd bonds $(0,1), (2,3), \dots$, then $\Delta t$ on even bonds $(1,2), (3,4), \dots$, then $\Delta t/2$ on odd bonds.
5. **MPS Transfer-Matrix Observables (`measure_mps_site_populations_transfer_matrix` — TODO 11)** & **Transport Driver (`run_tddmrg_transport_simulation` — TODO 12)**: Compute $\langle \hat{n}_k(t)\rangle$, wavepacket Mean-Squared Displacement $\text{MSD}(t) = \sum_k (k - k_0)^2\langle\hat{n}_k(t)\rangle$, and half-chain entanglement $S_{\text{vN}}(t)$.

In [ ]:
def initialize_localized_exciton_mps(n_sites: int, excited_site: int) -> list[np.ndarray]:
    '''
    TODO 7: Return a list of `n_sites` complex128 rank-3 MPS tensors of shape (1, 2, 1)
    representing |0 ... 0 1_{excited_site} 0 ... 0>.
    '''
    # TODO 7: Allocate (1, 2, 1) tensors and set A[0, 1, 0] = 1.0 at excited_site and A[0, 0, 0] = 1.0 elsewhere.
    raise NotImplementedError("Complete TODO 7: initialize_localized_exciton_mps")


def build_two_site_trotter_gates(
    j_hop: float = 1.0, v_int: float = 0.4, dt: float = 0.05
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 8: Construct the 4x4 two-site bond Hamiltonian h_bond and return
    (gate_half_4d, gate_full_4d), each of shape (2, 2, 2, 2), for dt/2 and dt.
    '''
    # TODO 8: Build 4x4 h_bond in basis {|00>, |01>, |10>, |11>}, compute expm(-1j * h_bond * (dt/2))
    #         and expm(-1j * h_bond * dt), and reshape each to (2, 2, 2, 2).
    raise NotImplementedError("Complete TODO 8: build_two_site_trotter_gates")


def apply_two_site_tebd_gate_and_svd(
    A_left: np.ndarray,
    A_right: np.ndarray,
    gate_4d: np.ndarray,
    max_bond_dim: int = 16,
    svd_cutoff: float = 1e-12,
) -> tuple[np.ndarray, np.ndarray, float]:
    '''
    TODO 9: Contract A_left (chi_L, 2, m) and A_right (m, 2, chi_R) with gate_4d (s1', s2', s1, s2),
    compress via SVD to max_bond_dim and svd_cutoff, split sqrt(S) symmetrically into new_left and new_right,
    and return (new_left, new_right, s_vn_entropy).
    '''
    # TODO 9: Contract theta = einsum("lsr,rtq->lstq", A_left, A_right), apply gate_4d,
    #         SVD-compress, compute s_vn_entropy, and return (new_left, new_right, s_vn_entropy).
    raise NotImplementedError("Complete TODO 9: apply_two_site_tebd_gate_and_svd")


def perform_strang_trotter_sweep(
    mps_tensors: list[np.ndarray],
    gate_half: np.ndarray,
    gate_full: np.ndarray,
    max_bond_dim: int = 16,
) -> tuple[list[np.ndarray], np.ndarray]:
    '''
    TODO 10: Perform one full second-order Strang-Trotter time step (odd bonds dt/2 -> even bonds dt -> odd bonds dt/2)
    and return (updated_mps_tensors, bond_entropies).
    '''
    # TODO 10: Apply apply_two_site_tebd_gate_and_svd on odd bonds (0, 2, ...) with gate_half,
    #          even bonds (1, 3, ...) with gate_full, and odd bonds (0, 2, ...) with gate_half.
    raise NotImplementedError("Complete TODO 10: perform_strang_trotter_sweep")


def measure_mps_site_populations_transfer_matrix(
    mps_tensors: list[np.ndarray],
) -> tuple[np.ndarray, float]:
    '''
    TODO 11: Compute local site populations <n_k> (for k = 0..n_sites-1) and total wavefunction norm
    using left-to-right 1D MPS transfer matrix contractions (without building the 2^L statevector).
    Return (populations, norm_val).
    '''
    # TODO 11: Contract transfer matrix E_{a,b} with n_op = diag(0, 1) at site `k` and identity I_2 elsewhere.
    raise NotImplementedError("Complete TODO 11: measure_mps_site_populations_transfer_matrix")


def run_tddmrg_transport_simulation(
    n_sites: int = 7,
    excited_site: int = 3,
    j_hop: float = 1.0,
    v_int: float = 0.4,
    dt: float = 0.05,
    n_steps: int = 60,
    max_bond_dim: int = 16,
) -> dict[str, np.ndarray]:
    '''
    TODO 12: Propagate the 1D chain for `n_steps` steps using TODOs 7-11 and return a dict with keys:
    'times', 'site_populations', 'msd', 'half_chain_entropy', 'norm_history'.
    '''
    # TODO 12: Initialize MPS, build Trotter gates, loop over steps recording populations, MSD(t), S_vN(L//2 - 1), and norm.
    raise NotImplementedError("Complete TODO 12: run_tddmrg_transport_simulation")

## Part 3: Hierarchical Equations of Motion (HEOM) for Non-Markovian Spin–Boson Dynamics (TODOs 13–18)

### Mathematical Specifications
For a spin qubit $\hat{H}_S = \frac{\epsilon}{2}\hat{\sigma}_z + \frac{\Delta}{2}\hat{\sigma}_x$ coupled via $\hat{S} = \hat{\sigma}_z$ to a Drude–Lorentz harmonic bath with reorganization energy $\lambda$, bath cutoff rate $\gamma$, and thermal energy $k_B T$, the high-temperature bath correlation function is $C(t) = (c_{\text{real}} + i c_{\text{imag}})e^{-\gamma t}$ with $c_{\text{real}} = 2\lambda k_B T$ and $c_{\text{imag}} = -\lambda\gamma$. The **Tanimura–Kubo HEOM hierarchy** ([QuTiP HEOM Guide](https://qutip.org/docs/4.7/guide/guide-heom.html)) propagates Auxiliary Density Operators $\hat{\rho}_n(t)$ ($n = 0, \dots, N_{\text{tier}}$):
$$\frac{d\hat{\rho}_n}{dt} = -i[\hat{H}_S, \hat{\rho}_n] - n\gamma\hat{\rho}_n - i[\hat{S}, \hat{\rho}_{n+1}] - i n\left(c_{\text{real}}[\hat{S}, \hat{\rho}_{n-1}] + i c_{\text{imag}}\{\hat{S}, \hat{\rho}_{n-1}\}\right)$$
where $\hat{\rho}_0(t)$ is the physical reduced density matrix of the spin subsystem and $\hat{\rho}_{n \ge 1}(t)$ encode non-Markovian system–bath entanglement and memory!

In [ ]:
def build_spin_boson_operators_and_bath_coeffs(
    epsilon: float = 0.5,
    delta: float = 1.0,
    reorg_lambda: float = 0.15,
    bath_gamma: float = 0.25,
    temperature_kt: float = 0.5,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 13: Construct 2x2 complex128 Pauli matrices sigma_x, sigma_z, system Hamiltonian
    H_s = 0.5*epsilon*sigma_z + 0.5*delta*sigma_x, coupling operator S_op = sigma_z,
    and Drude-Lorentz bath correlation coefficients c_real = 2*reorg_lambda*temperature_kt
    and c_imag = -reorg_lambda*bath_gamma.
    Return a dict with keys 'sigma_x', 'sigma_z', 'H_s', 'S_op', 'c_real', 'c_imag', 'gamma'.
    '''
    # TODO 13: Build sigma_x, sigma_z, H_s, S_op, c_real, c_imag, and return in a dictionary.
    raise NotImplementedError("Complete TODO 13: build_spin_boson_operators_and_bath_coeffs")


def compute_commutator_and_anticommutator(
    A: np.ndarray, B: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 14: Return ([A, B] = A @ B - B @ A, {A, B} = A @ B + B @ A).
    '''
    # TODO 14: Compute AB = A @ B and BA = B @ A, and return (AB - BA, AB + BA).
    raise NotImplementedError("Complete TODO 14: compute_commutator_and_anticommutator")


def evaluate_heom_tier_rhs(
    ados: np.ndarray,
    H_s: np.ndarray,
    S_op: np.ndarray,
    gamma: float,
    c_real: float,
    c_imag: float,
) -> np.ndarray:
    '''
    TODO 15: Evaluate the Tanimura-Kubo HEOM time derivative d(ados)/dt for `ados`
    of shape (n_tiers + 1, 2, 2) using `compute_commutator_and_anticommutator`.
    '''
    # TODO 15: Loop over tier n = 0..n_tiers, adding -i[H_s, rho_n] - n*gamma*rho_n,
    #          -i[S_op, rho_{n+1}] (if n < n_tiers), and -i*n*(c_real*[S_op, rho_{n-1}] + i*c_imag*{S_op, rho_{n-1}}) (if n > 0).
    raise NotImplementedError("Complete TODO 15: evaluate_heom_tier_rhs")


def rk4_step_heom_hierarchy(
    ados: np.ndarray,
    H_s: np.ndarray,
    S_op: np.ndarray,
    gamma: float,
    c_real: float,
    c_imag: float,
    dt: float,
) -> np.ndarray:
    '''
    TODO 16: Advance the HEOM auxiliary density operator hierarchy `ados` by `dt` using RK4.
    '''
    # TODO 16: Evaluate k1, k2, k3, k4 via evaluate_heom_tier_rhs and return ados + (dt/6)*(k1 + 2*k2 + 2*k3 + k4).
    raise NotImplementedError("Complete TODO 16: rk4_step_heom_hierarchy")


def extract_heom_observables_and_memory_norm(
    ados: np.ndarray, sigma_x: np.ndarray, sigma_z: np.ndarray
) -> dict[str, float]:
    '''
    TODO 17: From the current hierarchy `ados` of shape (n_tiers + 1, 2, 2), extract:
    'sigma_z' = Re(Tr(sigma_z @ rho_0)), 'sigma_x' = Re(Tr(sigma_x @ rho_0)),
    'trace' = Re(Tr(rho_0)), 'purity' = Re(Tr(rho_0 @ rho_0)), and
    'tier1_norm' = ||rho_1||_F (Frobenius norm of the n=1 auxiliary density operator).
    '''
    # TODO 17: Extract rho_0 = ados[0] and rho_1 = ados[1] and return the 5 quantities.
    raise NotImplementedError("Complete TODO 17: extract_heom_observables_and_memory_norm")


def run_heom_spin_boson_simulation(
    epsilon: float = 0.5,
    delta: float = 1.0,
    reorg_lambda: float = 0.15,
    bath_gamma: float = 0.25,
    temperature_kt: float = 0.5,
    n_tiers: int = 6,
    dt: float = 0.02,
    n_steps: int = 200,
) -> dict[str, np.ndarray]:
    '''
    TODO 18: Initialize ados[0] = |+><+| = [[0.5, 0.5], [0.5, 0.5]] (and ados[n>=1] = 0),
    propagate for `n_steps` steps using TODOs 13-17, and return a dict with keys:
    'times', 'sigma_z', 'sigma_x', 'trace_history', 'purity_history', 'tier1_norm'.
    '''
    # TODO 18: Combine TODOs 13, 16, and 17 to propagate the HEOM hierarchy and record time series.
    raise NotImplementedError("Complete TODO 18: run_heom_spin_boson_simulation")

## Part 4: Step-by-Step Checkpoint & Verification Suite

Run `run_section07_checkpoints()` below as you implement each `# TODO` function to test all 6 MCTDH functions, all 6 TD-DMRG functions, and all 6 HEOM functions!

In [2]:
def run_section07_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 18 #TODO functions in Section 07 (MCTDH + TD-DMRG + HEOM).'''
    passed = 0
    total_todos = 18

    try:
        # --- PART 1 CHECKS: 2D MCTDH (TODOs 1-6) ---
        q1, T1, dq1 = build_1d_sine_dvr_grid_and_kinetic(24, -4.0, 4.0, mass=1.0)
        assert q1.shape == (24,) and T1.shape == (24, 24) and np.allclose(T1, T1.T)
        evals_ho = np.sort(np.linalg.eigvalsh(T1 + 0.5 * np.diag(q1**2)))
        assert np.isclose(evals_ho[0], 0.5, atol=1e-4) and np.isclose(evals_ho[1], 1.5, atol=1e-4)
        passed += 1
        print("[PASS] TODO 1 : build_1d_sine_dvr_grid_and_kinetic (1D HO ground state = 0.50000 Eh)")

        A0, phi1_0, phi2_0 = initialize_2d_mctdh_state(q1, q1, n_spf=4)
        assert A0.shape == (4, 4) and np.linalg.norm(phi1_0.conj().T @ phi1_0 - np.eye(4)) < 1e-12
        passed += 1
        print("[PASS] TODO 2 : initialize_2d_mctdh_state")

        rho1, rho2, inv1, inv2 = compute_mctdh_density_and_regularized_inv(A0)
        assert rho1.shape == (4, 4) and np.isclose(rho1[0, 0].real, 1.0)
        passed += 1
        print("[PASS] TODO 3 : compute_mctdh_density_and_regularized_inv")

        h1_mat = T1 + 0.5 * np.diag(q1**2)
        h2_mat = T1 + 0.5 * (1.2**2) * np.diag(q1**2)
        q_mat = np.diag(q1)
        mf = compute_mctdh_projectors_and_mean_fields(A0, phi1_0, phi2_0, h1_mat, h2_mat, q_mat, q_mat)
        assert np.linalg.norm(mf["P1_orth"] @ phi1_0) < 1e-12
        passed += 1
        print("[PASS] TODO 4 : compute_mctdh_projectors_and_mean_fields")

        dA, dphi1, dphi2, e_init = evaluate_mctdh_dirac_frenkel_rhs(A0, phi1_0, phi2_0, h1_mat, h2_mat, q_mat, q_mat, c_coup=0.20)
        assert np.linalg.norm(phi1_0.conj().T @ dphi1) < 1e-12 and e_init > 1.0
        passed += 1
        print(f"[PASS] TODO 5 : evaluate_mctdh_dirac_frenkel_rhs (<H(0)> = {e_init:.6f} Eh)")

        mctdh_res = run_2d_mctdh_simulation(n_spf=4, n_grid=24, dt=0.02, n_steps=80, c_coup=0.20)
        assert np.max(np.abs(mctdh_res["norm_history"] - 1.0)) < 1e-5
        assert np.max(np.abs(mctdh_res["energy_history"] - mctdh_res["energy_history"][0])) < 1e-4
        passed += 1
        print("[PASS] TODO 6 : run_2d_mctdh_simulation (Norm & energy conserved, C(t) computed)")

        # --- PART 2 CHECKS: 1D TD-DMRG / MPS (TODOs 7-12) ---
        mps_0 = initialize_localized_exciton_mps(7, excited_site=3)
        assert len(mps_0) == 7 and mps_0[3][0, 1, 0] == 1.0
        passed += 1
        print("[PASS] TODO 7 : initialize_localized_exciton_mps")

        g_half, g_full = build_two_site_trotter_gates(j_hop=1.0, v_int=0.4, dt=0.05)
        assert g_half.shape == (2, 2, 2, 2) and np.linalg.norm(g_full.reshape(4, 4).conj().T @ g_full.reshape(4, 4) - np.eye(4)) < 1e-12
        passed += 1
        print("[PASS] TODO 8 : build_two_site_trotter_gates")

        nl, nr, s_b = apply_two_site_tebd_gate_and_svd(mps_0[3], mps_0[4], g_full, max_bond_dim=16)
        assert nl.shape[2] == nr.shape[0] and s_b > 0.0
        passed += 1
        print(f"[PASS] TODO 9 : apply_two_site_tebd_gate_and_svd (Bond S_vN = {s_b:.5f})")

        mps_1, ent_1 = perform_strang_trotter_sweep(mps_0, g_half, g_full, max_bond_dim=16)
        assert len(mps_1) == 7 and len(ent_1) == 6
        passed += 1
        print("[PASS] TODO 10: perform_strang_trotter_sweep")

        pops_1, norm_1 = measure_mps_site_populations_transfer_matrix(mps_1)
        assert np.isclose(np.sum(pops_1), 1.0, atol=1e-10) and np.isclose(norm_1, 1.0, atol=1e-10)
        passed += 1
        print(f"[PASS] TODO 11: measure_mps_site_populations_transfer_matrix (Total excitation = {np.sum(pops_1):.8f})")

        tddmrg_res = run_tddmrg_transport_simulation(n_sites=7, excited_site=3, dt=0.05, n_steps=40)
        assert tddmrg_res["msd"][-1] > 0.5 and np.max(np.abs(tddmrg_res["norm_history"] - 1.0)) < 1e-9
        passed += 1
        print(f"[PASS] TODO 12: run_tddmrg_transport_simulation (Final MSD = {tddmrg_res['msd'][-1]:.4f} sites^2)")

        # --- PART 3 CHECKS: HEOM SPIN-BOSON DYNAMICS (TODOs 13-18) ---
        sb = build_spin_boson_operators_and_bath_coeffs(epsilon=0.5, delta=1.0, reorg_lambda=0.15, bath_gamma=0.25, temperature_kt=0.5)
        assert sb["H_s"].shape == (2, 2) and np.isclose(sb["c_real"], 0.15) and np.isclose(sb["c_imag"], -0.0375)
        passed += 1
        print("[PASS] TODO 13: build_spin_boson_operators_and_bath_coeffs")

        comm_xz, acomm_xz = compute_commutator_and_anticommutator(sb["sigma_x"], sb["sigma_z"])
        assert np.linalg.norm(acomm_xz) < 1e-14 and np.linalg.norm(comm_xz) > 1.0
        passed += 1
        print("[PASS] TODO 14: compute_commutator_and_anticommutator")

        ados_0 = np.zeros((7, 2, 2), dtype=np.complex128)
        ados_0[0] = np.array([[0.5, 0.5], [0.5, 0.5]], dtype=np.complex128)
        rhs_0 = evaluate_heom_tier_rhs(ados_0, sb["H_s"], sb["S_op"], sb["gamma"], sb["c_real"], sb["c_imag"])
        assert abs(np.trace(rhs_0[0])) < 1e-15 and np.linalg.norm(rhs_0[1]) > 0.0
        passed += 1
        print("[PASS] TODO 15: evaluate_heom_tier_rhs")

        ados_1 = rk4_step_heom_hierarchy(ados_0, sb["H_s"], sb["S_op"], sb["gamma"], sb["c_real"], sb["c_imag"], dt=0.02)
        assert np.isclose(np.trace(ados_1[0]).real, 1.0, atol=1e-14)
        passed += 1
        print("[PASS] TODO 16: rk4_step_heom_hierarchy")

        obs_1 = extract_heom_observables_and_memory_norm(ados_1, sb["sigma_x"], sb["sigma_z"])
        assert np.isclose(obs_1["trace"], 1.0, atol=1e-14) and obs_1["tier1_norm"] > 0.0
        passed += 1
        print(f"[PASS] TODO 17: extract_heom_observables_and_memory_norm (||rho_1||_F = {obs_1['tier1_norm']:.5f})")

        heom_res = run_heom_spin_boson_simulation(n_tiers=6, dt=0.02, n_steps=150)
        assert np.max(np.abs(heom_res["trace_history"] - 1.0)) < 1e-12 and heom_res["purity_history"][-1] < 0.95
        passed += 1
        print(f"[PASS] TODO 18: run_heom_spin_boson_simulation (Final purity = {heom_res['purity_history'][-1]:.4f}, max |Tr-1| = {np.max(np.abs(heom_res['trace_history']-1.0)):.2e})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 07 (MCTDH + TD-DMRG + HEOM) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section07_checkpoints()
print("Section 07 checkpoint suite loaded. Implement TODOs 1-18 and call `run_section07_checkpoints()`!")

Section 07 checkpoint suite loaded. Implement TODOs 1-18 and call `run_section07_checkpoints()`!
